In [ ]:
import random

import carla

In [ ]:
# ============================================================
# Configuration
# ============================================================

CARLA_HOST = "localhost"
CARLA_PORT = 2000
CARLA_TIMEOUT_SECONDS = 10.0

TRAFFIC_MANAGER_PORT = 8000

NUMBER_OF_VEHICLES = 30
RANDOM_SEED = 42

FOLLOWING_DISTANCE_METERS = 3.0
SPEED_DIFFERENCE_PERCENT = 10.0

VEHICLE_FILTER = "vehicle.*"

In [ ]:
# ============================================================
# CARLA Connection
# ============================================================

def connect_to_carla() -> tuple[carla.Client, carla.World]:
    """Connect to the CARLA server and return the client and world."""
    client = carla.Client(
        CARLA_HOST,
        CARLA_PORT,
    )

    client.set_timeout(
        CARLA_TIMEOUT_SECONDS
    )

    world = client.get_world()

    return client, world

In [ ]:
def get_vehicle_blueprints(
    blueprint_library: carla.BlueprintLibrary,
) -> list[carla.ActorBlueprint]:
    """Return all available vehicle blueprints."""

    blueprints = list(
        blueprint_library.filter(VEHICLE_FILTER)
    )

    if not blueprints:
        raise RuntimeError(
            "No vehicle blueprints are available."
        )

    return blueprints

In [ ]:
def get_spawn_points(
    world: carla.World,
) -> list[carla.Transform]:
    """Return shuffled vehicle spawn points."""

    spawn_points = list(
        world.get_map().get_spawn_points()
    )

    if not spawn_points:
        raise RuntimeError(
            "No vehicle spawn points are available."
        )

    random.shuffle(spawn_points)

    return spawn_points

In [ ]:
def spawn_vehicles(
    world: carla.World,
    vehicle_blueprints: list[carla.ActorBlueprint],
    spawn_points: list[carla.Transform],
    number_of_vehicles: int,
) -> list[carla.Vehicle]:
    """
    Spawn multiple vehicles using random blueprints
    and shuffled spawn points.
    """

    vehicles: list[carla.Vehicle] = []

    spawn_count = min(
        number_of_vehicles,
        len(spawn_points),
    )

    for index in range(spawn_count):

        vehicle_bp = random.choice(
            vehicle_blueprints
        )

        spawn_point = spawn_points[index]

        vehicle = world.try_spawn_actor(
            vehicle_bp,
            spawn_point,
        )

        if vehicle is None:
            print(
                f"[WARNING] Failed to spawn vehicle "
                f"at spawn point {index + 1}."
            )
            continue

        vehicles.append(vehicle)

        print(
            f"[{len(vehicles):02d}] "
            f"ID={vehicle.id} "
            f"Type={vehicle.type_id}"
        )

    return vehicles

In [ ]:
# ============================================================
# Traffic Manager
# ============================================================

def configure_traffic_manager(
    traffic_manager: carla.TrafficManager,
) -> None:
    """Configure global Traffic Manager behavior."""

    traffic_manager.set_global_distance_to_leading_vehicle(
        FOLLOWING_DISTANCE_METERS
    )

    traffic_manager.global_percentage_speed_difference(
        SPEED_DIFFERENCE_PERCENT
    )


In [ ]:
# ============================================================
# Autopilot
# ============================================================

def enable_autopilot(
    vehicles: list[carla.Vehicle],
) -> None:
    """Enable Traffic Manager control for all vehicles."""

    for vehicle in vehicles:
        if vehicle.is_alive:
            vehicle.set_autopilot(
                True,
                TRAFFIC_MANAGER_PORT,
            )


In [ ]:
# ============================================================
# Cleanup
# ============================================================

def destroy_vehicles(
    vehicles: list[carla.Vehicle],
) -> None:
    """Destroy all vehicles created by this script."""

    for vehicle in vehicles:
        if vehicle.is_alive:
            vehicle.destroy()

In [ ]:
# ============================================================
# Main
# ============================================================

def main() -> None:

    random.seed(RANDOM_SEED)

    client, world = connect_to_carla()

    blueprint_library = (
        world.get_blueprint_library()
    )

    traffic_manager = (
        client.get_trafficmanager(
            TRAFFIC_MANAGER_PORT
        )
    )

    vehicle_blueprints = (
        get_vehicle_blueprints(
            blueprint_library
        )
    )

    spawn_points = get_spawn_points(
        world
    )

    print("=== TRAFFIC GENERATION ===")
    print(
        f"Available vehicle blueprints: "
        f"{len(vehicle_blueprints)}"
    )
    print(
        f"Available spawn points: "
        f"{len(spawn_points)}"
    )
    print(
        f"Requested vehicles: "
        f"{NUMBER_OF_VEHICLES}"
    )
    print()

    configure_traffic_manager(
        traffic_manager
    )

    vehicles = spawn_vehicles(
        world,
        vehicle_blueprints,
        spawn_points,
        NUMBER_OF_VEHICLES,
    )

    enable_autopilot(
        vehicles
    )

    print()
    print("=== TRAFFIC RUNNING ===")
    print(
        f"Active vehicles: "
        f"{len(vehicles)}"
    )
    print(
        f"Following distance: "
        f"{FOLLOWING_DISTANCE_METERS} m"
    )
    print(
        f"Speed difference: "
        f"{SPEED_DIFFERENCE_PERCENT}%"
    )
    print()
    print(
        "Press ENTER to stop the simulation..."
    )

    try:
        input()
    finally:
        print()
        print("=== CLEANUP ===")

        destroy_vehicles(
            vehicles
        )

        print(
            f"Destroyed vehicles: "
            f"{len(vehicles)}"
        )


if __name__ == "__main__":
    main()